# External transfer: TCGA-trained model on a GEO cohort

This notebook trains an `E2MModel` on TCGA lung adenocarcinoma (`LUAD`) and applies it to **GSE31210** (Okayama et al.), an independent lung-adenocarcinoma microarray cohort, to demonstrate cross-cohort prediction end to end.

**Requirements:** network access and `GEOparse` for the GEO download:

```bash
pip install GEOparse
```

**Caveat:** GSE31210 is Affymetrix microarray data, on a different measurement scale from the TCGA STAR counts the model is trained on. Sections 1-4 ignore this gap, and show the *mechanics* - align genes by symbol, then predict. Section 5 shows the integration used in the manuscript (within-sample ranks and a retrained model) and compares the integration and the metrics before and after.

In [6]:
from pathlib import Path
import pandas as pd

from e2m import Dataset, E2MModel

DATA_DIR = Path("e2m_data")
RESULT_DIR = Path("results/external_gse31210")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Train the mutation model on TCGA-LUAD

This downloads and prepares TCGA-LUAD on the first run (cached under `DATA_DIR`) and fits the multitask network with the default 15% early-stopping split and no subsequent refit on all samples. TMB is not needed here, so it is skipped.

In [7]:
tcga = Dataset.from_tcga(["LUAD"], data_dir=DATA_DIR, with_tmb=False)
model = E2MModel().fit(tcga)
len(tcga.expression), len(model.targets)

(510, 400)

## 2. Download and prepare the external cohort

`GEOparse` downloads the series matrix and its platform annotation. We map probes to gene symbols, average duplicate symbols, and return a samples-by-genes dataframe.

If the GEO download fails (usually a network issue), rerun the cell, or download [GSE31210_family.soft.gz](https://ftp.ncbi.nlm.nih.gov/geo/series/GSE31nnn/GSE31210/soft/GSE31210_family.soft.gz) manually into `e2m_data/geo/` next to this notebook.

In [8]:
def _symbol_column(annotation_table):
    for name in ("Gene Symbol", "Gene symbol", "GENE_SYMBOL", "Symbol", "gene_assignment"):
        if name in annotation_table.columns:
            return name
    raise ValueError(
        f"No gene-symbol column in the platform annotation: {list(annotation_table.columns)}"
    )


def load_gse31210(cache_dir):
    """Download GSE31210, map probes to symbols, return a samples-by-genes frame."""
    import GEOparse

    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    try:
        gse = GEOparse.get_GEO(geo="GSE31210", destdir=str(cache_dir), silent=True)
    except OSError as err:
        url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE31nnn/GSE31210/soft/GSE31210_family.soft.gz"
        raise RuntimeError(
            f"GSE31210 download from GEO failed ({err}). This is usually a network issue: rerun the cell, "
            f"or download {url} manually into {cache_dir.resolve()} and rerun."
        ) from err
    probes = gse.pivot_samples("VALUE")  # probes x samples

    platform = list(gse.gpls.values())[0]
    annotation = platform.table.set_index("ID")
    symbols = annotation[_symbol_column(annotation)].reindex(probes.index).astype(str)
    symbols = symbols.str.split(r"\s*///\s*").str[0].str.strip()

    keep = symbols.notna() & ~symbols.isin({"", "nan", "---"})
    probes = probes.loc[keep]
    probes.index = symbols[keep].values
    expression = probes.apply(pd.to_numeric, errors="coerce").groupby(level=0).mean().T
    expression.index.name = "sample"
    return expression

In [10]:
external = load_gse31210(DATA_DIR / "geo")
external.shape

c:\Users\KerenYlab.MEDICINE\AppData\Local\anaconda3\envs\e2m\lib\site-packages\GEOparse\GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")


(246, 22880)

## 3. Predict mutation probabilities on the external cohort

Input genes are aligned to the model's training features by symbol. Missing features are filled with their training means. 
Alternatively, you can filter genes in advance, by uncommenting below:

In [11]:
# Optional: train on the genes both cohorts measure, so no features are filled in at prediction
# shared = tcga.expression.columns.intersection(external.columns)
# tcga = Dataset(tcga.expression[shared], mutations=tcga.mutations, cancer=tcga.cancer)
# model = E2MModel().fit(tcga)

# min_feature_overlap=0.0 allows predicting when only part of the model's genes are measured
probabilities = model.predict(external, min_feature_overlap=0.0)
probabilities.to_csv(RESULT_DIR / "gse31210_probabilities.csv")
probabilities.iloc[:5, :8]

,TP53,TTN,MUC16,CSMD3,RYR2,LRP1B,USH2A,ZFHX4
sample,,,,,,,,
GSM773540,0.300276,0.365536,0.511472,0.347839,0.464223,0.398669,0.355964,0.375790
GSM773541,0.360612,0.410475,0.577524,0.379268,0.512332,0.428126,0.381860,0.381334
GSM773542,0.295969,0.358855,0.516143,0.332272,0.468811,0.398591,0.344379,0.359927
GSM773543,0.274515,0.357226,0.506150,0.316282,0.459694,0.388010,0.342579,0.365443
GSM773544,0.306804,0.364122,0.525811,0.339885,0.481276,0.413989,0.354611,0.357578


## 4. Evaluate against the reported EGFR and KRAS status

To evaluate predictions on an external cohort, you need its mutation labels as a samples-by-targets 0/1 table, with target names matching `model.targets`. GSE31210 reports each tumor's EGFR and KRAS status, so we build that table below and score these two targets.

In [13]:
def load_alteration_status(cache_dir):
    """EGFR / KRAS status reported for each GSE31210 tumor, from the cached download."""
    import GEOparse

    gse = GEOparse.get_GEO(geo="GSE31210", destdir=str(Path(cache_dir)), silent=True)
    reported = {}
    for name, gsm in gse.gsms.items():
        for field in gsm.metadata.get("characteristics_ch1", []):
            key, _, value = field.partition(":")
            if key.strip() == "gene alteration status":
                reported[name] = value.strip()

    status = pd.Series(reported, name="alteration").sort_index()
    return pd.DataFrame(
        {
            "EGFR": (status == "EGFR mutation +").astype(int),
            "KRAS": (status == "KRAS mutation +").astype(int),
        },
        index=status.index,
    )


status = load_alteration_status(DATA_DIR / "geo")
status.sum()

c:\Users\KerenYlab.MEDICINE\AppData\Local\anaconda3\envs\e2m\lib\site-packages\GEOparse\GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")


EGFR    127
KRAS     20
dtype: int64

In [14]:
from sklearn.metrics import average_precision_score, roc_auc_score

from e2m.evaluation import normalized_auprc


def score_targets(probabilities, targets=("EGFR", "KRAS")):
    """Ranking metrics for the reported targets, over the samples both frames cover."""
    scored = status.index.intersection(probabilities.index)
    rows = []
    for target in targets:
        truth = status.loc[scored, target]
        probability = probabilities.loc[scored, target]
        prevalence = float(truth.mean())
        auprc = float(average_precision_score(truth, probability))
        rows.append(
            {
                "target": target,
                "n_samples": len(scored),
                "n_positive": int(truth.sum()),
                "prevalence": prevalence,
                "auprc": auprc,
                "normalized_auprc": normalized_auprc(auprc, prevalence),
                "roc_auc": float(roc_auc_score(truth, probability)),
            }
        )
    return pd.DataFrame(rows).set_index("target")


external_metrics = score_targets(probabilities)
external_metrics.to_csv(RESULT_DIR / "gse31210_external_metrics.csv")
external_metrics.round(3)

,n_samples,n_positive,prevalence,auprc,normalized_auprc,roc_auc
target,,,,,,
EGFR,226,127,0.562,0.626,0.146,0.603
KRAS,226,20,0.088,0.210,0.133,0.663


`normalized_auprc` is the same quantity the cross-validation tables report, `(AUPRC - prevalence) / (1 - prevalence)`, where 0 is chance at the given prevalence and 1 is a perfect ranking. 

## 5. Within-sample ranks and a retrained model

Sections 3-4 fed microarray values to a model trained on RNA-seq. The manuscript's external analyses ([E2M-analyses, Runs/External](https://github.com/asafpinhasitechnion/E2M-analyses/tree/main/Runs/External)) instead keep the genes shared by TCGA and the external cohort, convert both to within-sample gene ranks, and train a new model on the ranked TCGA. Each sample is ranked on its own, so samples can be predicted one at a time, without a cohort-level correction.

In [ ]:
def sample_rank(expression):
    """Rank the genes within each sample, scaled to (0, 1] (`sample_rank` in E2M-analyses, Runs/External/src/integration.py)."""
    return expression.rank(axis=1, method="average") / expression.shape[1]

In [ ]:
def external_dataset(expression):
    """Wrap an external expression matrix as a Dataset carrying the reported labels."""
    labelled = expression.index.intersection(status.index)
    return Dataset(
        expression.loc[labelled],
        mutations=status.loc[labelled],
        cancer=pd.Series("GSE31210", index=labelled),
    )


# Genes measured in both cohorts, ranked within each sample
shared = tcga.expression.columns.intersection(external.columns).sort_values()
tcga_ranked = Dataset(sample_rank(tcga.expression[shared].fillna(0.0)), mutations=tcga.mutations, cancer=tcga.cancer)
gse31210 = external_dataset(external)
gse31210_ranked = external_dataset(sample_rank(external[shared].fillna(0.0)))

# TCGA is transformed too, so a new model is trained on the ranked TCGA
model_ranked = E2MModel().fit(tcga_ranked)

tcga_ranked, gse31210_ranked

### Integration before and after

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler


def neighbour_mixing(values, labels, n_neighbors=30):
    """Mean fraction of each sample's neighbours that come from the other cohort."""
    finder = NearestNeighbors(n_neighbors=n_neighbors + 1).fit(values)
    neighbours = finder.kneighbors(values, return_distance=False)[:, 1:]  # drop self
    return float((labels[neighbours] != labels[:, None]).mean())


def integration_panel(variants, n_neighbors=30):
    """PCA of TCGA + external, one panel per (TCGA, external) pair, coloured by cohort."""
    fig, axes = plt.subplots(1, len(variants), figsize=(5.4 * len(variants), 4.4), squeeze=False)
    for ax, (title, (reference, ext)) in zip(axes[0], variants.items()):
        shared = [gene for gene in ext.columns if gene in set(reference.columns)]
        stacked = np.vstack([
            reference[shared].to_numpy(dtype=float),
            ext[shared].to_numpy(dtype=float),
        ])
        labels = np.array(["TCGA-LUAD"] * len(reference) + ["GSE31210"] * len(ext))

        scaled = StandardScaler().fit_transform(stacked)
        embedding = PCA(n_components=min(20, *scaled.shape) , random_state=0).fit_transform(scaled)

        for cohort, colour in (("TCGA-LUAD", "#4C72B0"), ("GSE31210", "#DD8452")):
            mask = labels == cohort
            ax.scatter(embedding[mask, 0], embedding[mask, 1], s=10, alpha=0.6,
                       c=colour, label=cohort, linewidths=0)
        mixing = neighbour_mixing(embedding, labels, n_neighbors)
        # With cohorts of unequal size, perfect mixing is 2p(1-p), not 1.
        share = len(ext) / len(labels)
        ax.set_title(f"{title}\nneighbour mixing {mixing:.2f} (balanced {2 * share * (1 - share):.2f})")
        ax.set_xlabel("PC-1")
        ax.set_ylabel("PC-2")
        ax.set_xticks([])
        ax.set_yticks([])
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
    axes[0][0].legend(frameon=False, loc="best", fontsize=9)
    fig.tight_layout()
    plt.show()


integration_panel(
    {
        "Before": (tcga.expression, gse31210.expression),
        "Within-sample ranks": (tcga_ranked.expression, gse31210_ranked.expression),
    },
)

### Metrics before and after

Same labels as section 4: the original input with the section 1 model, and the ranked input with the model trained on ranked TCGA.

In [ ]:
predictions = {
    "uncorrected": model.predict(gse31210, min_feature_overlap=0.0),
    "sample-ranked": model_ranked.predict(gse31210_ranked),
}
for name, frame in predictions.items():
    frame.to_csv(RESULT_DIR / f"gse31210_probabilities_{name.replace('-', '_')}.csv")

comparison = pd.concat(
    {name: score_targets(frame) for name, frame in predictions.items()},
    names=["input"],
)
comparison.to_csv(RESULT_DIR / "gse31210_metrics_before_after.csv")
comparison.round(3)